# Setup Guide

## Context
The model was trained on Roboflow, but exporting trained weights requires a paid plan.
This notebook downloads the **annotated dataset** from Roboflow and re-trains locally to produce a usable `.pt` model file.

---

## Requirements

### Python
Recommended: **Python 3.10 or 3.11**

### GPU (Strongly Recommended)
Training uses `device=0` (GPU). Without a CUDA-capable NVIDIA GPU training will be extremely slow.

---

## 1. Install PyTorch (with CUDA)

Go to [https://pytorch.org/get-started/locally](https://pytorch.org/get-started/locally) and pick the right command for your CUDA version.

Example for **CUDA 12.1**:
```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

Verify GPU is available after install:
```python
import torch
print(torch.cuda.is_available())  # Should print True
```

---

## 2. Install Required Libraries

```bash
pip install ultralytics roboflow
```

| Package | Purpose |
|---|---|
| `ultralytics` | YOLO v8–v12 training & inference |
| `roboflow` | Downloading the annotated dataset |

---

## 3. Dataset Info (from Roboflow)

| Property | Value |
|---|---|
| Total images | 36,560 (after tiling) |
| Train / Valid / Test | 82% / 12% / 6% |
| Original split | 3,388 / 968 / 484 images |

**Preprocessing applied by Roboflow (already baked into downloaded images):**
- Dynamic Crop to annotated mine region (Isolate Objects)
- Resize to 640×640 (stretch)
- Auto-Contrast (Adaptive Equalization)
- Auto-Orient

**Augmentations baked in (2 outputs per training image):**
- Flip horizontal + vertical
- 90° rotate (CW, CCW, 180°)
- Rotation ±15°, Shear ±10°H/V
- Crop 0–20%, Brightness ±15%, Exposure ±10%
- Saturation ±25%, Hue ±15°
- Blur ≤2.5px, Noise ±0.1%, Motion Blur 30px, Camera Gain σ=0.05
- Grayscale (applied to 25% of images)

> Because augmentations are already baked into the downloaded dataset, the training cell applies a matching but lighter set on-the-fly each epoch.

---

## 4. Notes

- `yolo26n.pt` is the base pretrained YOLO nano model — downloaded automatically on first run.
- After training, the best weights are saved to `runs/detect/trainX/weights/best.pt` — that is your final model.
- If you get a CUDA out-of-memory error, lower `batch` from 32 → 16 in the training cell.
- For better accuracy at the cost of speed/VRAM, swap `yolo26n.pt` → `yolo26s.pt` or `yolo26m.pt`.

In [ ]:
from roboflow import Roboflow

In [ ]:
rf = Roboflow(api_key="#APIKEY#")
project = rf.workspace("#YOURWORKSPACE#").project("#YOURPROJECT#")
dataset = project.version(1).download("yolo26")
print(dataset.location)

In [ ]:
from ultralytics import YOLO

# yolo26n = fastest/smallest | yolo26s = better accuracy | yolo26m/l = best accuracy (needs more VRAM)
model = YOLO("yolo26n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",

    # --- Core ---
    epochs=100,
    imgsz=640,
    device=0,
    batch=32,             # lower to 16/8/4 if you get CUDA out-of-memory errors

    # --- Optimizer ---
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    weight_decay=0.0005,

    # --- Regularization ---
    dropout=0.1,
    label_smoothing=0.1,

    # --- On-the-fly augmentation (mirrors Roboflow config) ---
    fliplr=0.5,           # horizontal flip ON
    flipud=0.5,           # vertical flip ON
    degrees=15.0,         # rotation ±15°
    shear=10.0,           # shear ±10°H / ±10°V
    scale=0.2,            # crop 0–20% zoom
    hsv_h=0.04,           # hue ±15° (15/360 ≈ 0.04)
    hsv_s=0.25,           # saturation ±25%
    hsv_v=0.15,           # brightness ±15%

    # --- Early stopping & checkpointing ---
    patience=50,
    save_period=10,

    # --- Output ---
    plots=True,
    val=True,
)

#Validating best checkpoint on test set

In [ ]:
# Validate the best checkpoint on the test set
metrics = model.val(split="test")  # change to split="val" if no test split exists

print(f"mAP50:     {metrics.box.map50:.4f}")   # main metric — target >0.80
print(f"mAP50-95:  {metrics.box.map:.4f}")     # stricter IoU range
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

#Training Summary:

In [ ]:
import pandas as pd
from pathlib import Path

# Find the latest training run folder
run_dir = Path(results.save_dir)
csv_path = run_dir / "results.csv"

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

total_epochs = len(df)
best_epoch = df["metrics/mAP50(B)"].idxmax() + 1  # 1-based
best_map50 = df["metrics/mAP50(B)"].max()

print(f"Run folder  : {run_dir}")
print(f"Epochs ran  : {total_epochs}  (stopped early if < 100)")
print(f"Best epoch  : {best_epoch}  →  best.pt was saved from this epoch")
print(f"Best mAP50  : {best_map50:.4f}")
print(f"\nbest.pt path: {run_dir / 'weights' / 'best.pt'}")
print(f"last.pt path: {run_dir / 'weights' / 'last.pt'}")
print("\nbest.pt  = weights from the single epoch with highest mAP50 — USE THIS")
print("last.pt  = weights from the final epoch regardless of performance — ignore unless resuming")

Exporting pt. to .onnx file format:

In [ ]:
import shutil

# .pt weights do NOT have imgsz baked in size only matters at inference.
# ONNX exports DO bake the input shape into the graph, making each version truly different.

best_pt = Path(results.save_dir) / "weights" / "best.pt"
out_dir = best_pt.parent

for size in [160, 320, 640]:
    m = YOLO(str(best_pt))
    export_path = m.export(format="onnx", imgsz=size, dynamic=False)
    dest = out_dir / f"yolov26pfm1_{size}.onnx"
    shutil.move(export_path, dest)
    print(f"Exported → {dest}")